# Merge Italian Bank Office Cross-Sections into a Panel (1885, 1897, 1907)

This notebook merges three cross-section dataframes of Italian bank offices into a
longitudinal panel dataset, linked by **municipality**.

**Workflow:**
1. Upload your three files (CSV or Excel)
2. Select which column contains the municipality name in each file
3. Automatic **perfect matching** of identical municipality names
4. **Fuzzy matching** of remaining unmatched municipalities, reviewed interactively by you
5. Export the final panel dataset

Historical municipality name changes (e.g. *Caltanissetta* vs *Caltanisetta*,
*Girgenti* vs *Agrigento*) are handled via fuzzy matching with your manual review.

## 1. Install dependencies and imports

In [ ]:
!pip install -q thefuzz python-Levenshtein openpyxl xlrd

import pandas as pd
import numpy as np
import re
import unicodedata
import json
import os
from thefuzz import fuzz, process
from google.colab import files
from IPython.display import display, HTML, clear_output

print("All dependencies loaded.")

## 2. Upload your three cross-section files

Upload CSV or Excel files for each benchmark year. You will be prompted once per year.

In [ ]:
def load_file(year):
    """Upload and load a single cross-section file."""
    print(f"\n{'='*60}")
    print(f"  Upload the file for year {year}")
    print(f"{'='*60}")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError(f"No file uploaded for {year}.")
    filename = list(uploaded.keys())[0]
    if filename.endswith('.csv'):
        # Try common encodings for Italian historical data
        for enc in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
            try:
                df = pd.read_csv(filename, encoding=enc)
                break
            except (UnicodeDecodeError, Exception):
                continue
        else:
            df = pd.read_csv(filename, encoding='latin-1', errors='replace')
    elif filename.endswith(('.xls', '.xlsx')):
        xls = pd.ExcelFile(filename)
        if len(xls.sheet_names) > 1:
            print(f"\nSheets found: {xls.sheet_names}")
            sheet = input(f"Which sheet for {year}? (press Enter for '{xls.sheet_names[0]}'): ").strip()
            sheet = sheet if sheet else xls.sheet_names[0]
        else:
            sheet = xls.sheet_names[0]
        df = pd.read_excel(filename, sheet_name=sheet)
    else:
        raise ValueError(f"Unsupported file type: {filename}")

    print(f"\nLoaded {filename}: {df.shape[0]} rows, {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    display(df.head(3))
    return df, filename

# Load all three years
df_1885, fname_1885 = load_file(1885)
df_1897, fname_1897 = load_file(1897)
df_1907, fname_1907 = load_file(1907)

## 3. Select the municipality column for each year

Identify which column contains the municipality (comune) name in each dataframe.

In [ ]:
def select_municipality_column(df, year):
    """Let user pick the municipality column."""
    cols = list(df.columns)
    print(f"\nColumns in {year} data:")
    for i, c in enumerate(cols):
        print(f"  [{i}] {c}")

    # Try to auto-detect
    candidates = [c for c in cols if any(kw in c.lower() for kw in
                  ['comune', 'municip', 'city', 'citta', 'città', 'sede',
                   'localit', 'luogo', 'town'])]
    suggestion = f" (suggested: '{candidates[0]}')" if candidates else ""
    idx = input(f"Enter column number for municipality in {year}{suggestion}: ").strip()
    col = cols[int(idx)]
    print(f"  -> Selected: '{col}'")
    return col

muni_col_1885 = select_municipality_column(df_1885, 1885)
muni_col_1897 = select_municipality_column(df_1897, 1897)
muni_col_1907 = select_municipality_column(df_1907, 1907)

## 4. Normalize municipality names

Standardize names to maximize perfect matches:
- Lowercase
- Remove accents
- Strip punctuation and extra whitespace
- Normalize common Italian prefixes (S. -> San/Santo/Santa etc.)

In [ ]:
def normalize_municipality(name):
    """Normalize an Italian municipality name for matching."""
    if pd.isna(name):
        return ""
    s = str(name).strip()
    # Lowercase
    s = s.lower()
    # Remove accents
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    # Normalize common abbreviations
    # S. / S  at word boundary -> san (will match san/sant/santo/santa via fuzzy)
    s = re.sub(r"\bs\.\s*", "san ", s)
    # Remove punctuation except spaces
    s = re.sub(r"[^a-z0-9\s]", "", s)
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Apply normalization
df_1885['_muni_norm'] = df_1885[muni_col_1885].apply(normalize_municipality)
df_1897['_muni_norm'] = df_1897[muni_col_1897].apply(normalize_municipality)
df_1907['_muni_norm'] = df_1907[muni_col_1907].apply(normalize_municipality)

# Extract unique municipality lists
munis_1885 = sorted(df_1885['_muni_norm'].unique().tolist())
munis_1897 = sorted(df_1897['_muni_norm'].unique().tolist())
munis_1907 = sorted(df_1907['_muni_norm'].unique().tolist())

# Remove empty strings
munis_1885 = [m for m in munis_1885 if m]
munis_1897 = [m for m in munis_1897 if m]
munis_1907 = [m for m in munis_1907 if m]

print(f"Unique municipalities: 1885={len(munis_1885)}, 1897={len(munis_1897)}, 1907={len(munis_1907)}")
print(f"\nSample normalized names (1885): {munis_1885[:10]}")

## 5. Build the master municipality list via perfect + fuzzy matching

**Strategy:**
1. Pool all normalized names from the three years
2. Perfect matches (identical normalized strings) are grouped automatically
3. Remaining unmatched names are compared via fuzzy matching
4. Fuzzy candidates above a threshold are presented to you for review

Each accepted group of names gets a single **canonical municipality ID** used in the panel.

In [ ]:
from collections import defaultdict

# ------------------------------------------------------------------
# 5a. Build tagged name list: (normalized_name, year, original_name)
# ------------------------------------------------------------------
tagged_names = []
for m in df_1885[[muni_col_1885, '_muni_norm']].drop_duplicates().itertuples(index=False):
    tagged_names.append((m[1], 1885, str(m[0]).strip()))
for m in df_1897[[muni_col_1897, '_muni_norm']].drop_duplicates().itertuples(index=False):
    tagged_names.append((m[1], 1897, str(m[0]).strip()))
for m in df_1907[[muni_col_1907, '_muni_norm']].drop_duplicates().itertuples(index=False):
    tagged_names.append((m[1], 1907, str(m[0]).strip()))

# Remove entries with empty normalized names
tagged_names = [(n, y, o) for n, y, o in tagged_names if n]

print(f"Total tagged entries: {len(tagged_names)}")

# ------------------------------------------------------------------
# 5b. Perfect matching: group by identical normalized name
# ------------------------------------------------------------------
perfect_groups = defaultdict(list)  # norm_name -> [(year, original_name), ...]
for norm, year, orig in tagged_names:
    perfect_groups[norm].append((year, orig))

# Separate into matched (appear in 2+ years) and unmatched
perfect_matched = {}   # norm_name -> [(year, orig), ...]
unmatched = {}          # norm_name -> [(year, orig), ...]

for norm, entries in perfect_groups.items():
    years_present = set(y for y, _ in entries)
    if len(years_present) >= 2:
        perfect_matched[norm] = entries
    else:
        unmatched[norm] = entries

# Also include multi-year groups in matched
# Single-year names go to unmatched for fuzzy matching
print(f"\nPerfect matches (same name in 2+ years): {len(perfect_matched)} groups")
print(f"Unmatched (appear in only 1 year): {len(unmatched)} names")
print(f"\nSample perfect matches:")
for i, (k, v) in enumerate(list(perfect_matched.items())[:5]):
    years = sorted(set(y for y, _ in v))
    print(f"  '{k}' -> years {years}")

## 6. Fuzzy matching of unmatched municipalities

For each unmatched name, we search for the best fuzzy match among:
- Other unmatched names from **different years**
- Already-matched group names (to merge a missing year into an existing group)

You will review each proposed match interactively.

In [ ]:
# ------------------------------------------------------------------
# 6a. Compute fuzzy candidates
# ------------------------------------------------------------------
FUZZY_THRESHOLD = 75  # Minimum score to propose a match

# Build target lists for fuzzy matching
# For each unmatched name, find candidates from OTHER years
unmatched_by_year = defaultdict(list)
for norm, entries in unmatched.items():
    for year, orig in entries:
        unmatched_by_year[year].append(norm)

# All names that could be targets (perfect-matched names + unmatched from other years)
all_target_names = list(perfect_matched.keys()) + list(unmatched.keys())

# For each unmatched name, find top fuzzy matches from different years
fuzzy_candidates = []  # (name_a, year_a, orig_a, name_b, year_b, orig_b, score)

already_processed = set()

for norm_a, entries_a in unmatched.items():
    years_a = set(y for y, _ in entries_a)

    # Build targets: names NOT in the same year-only group
    targets = []
    for norm_b, entries_b in unmatched.items():
        if norm_b == norm_a:
            continue
        years_b = set(y for y, _ in entries_b)
        # Must have at least one different year
        if years_b != years_a or years_a != years_b:
            targets.append(norm_b)

    # Also check against perfect-matched names
    for norm_b in perfect_matched:
        targets.append(norm_b)

    if not targets:
        continue

    # Get top 3 matches
    matches = process.extract(norm_a, targets, scorer=fuzz.ratio, limit=3)
    for match_name, score, _ in matches:
        if score >= FUZZY_THRESHOLD:
            pair = tuple(sorted([norm_a, match_name]))
            if pair not in already_processed:
                already_processed.add(pair)
                # Get original names for display
                if match_name in unmatched:
                    entries_b = unmatched[match_name]
                else:
                    entries_b = perfect_matched[match_name]

                fuzzy_candidates.append({
                    'norm_a': norm_a,
                    'entries_a': entries_a,
                    'norm_b': match_name,
                    'entries_b': entries_b,
                    'score': score
                })

# Sort by score descending (best matches first)
fuzzy_candidates.sort(key=lambda x: x['score'], reverse=True)

print(f"Found {len(fuzzy_candidates)} fuzzy match candidates (score >= {FUZZY_THRESHOLD})")
print(f"\nTop 10 candidates:")
for i, c in enumerate(fuzzy_candidates[:10]):
    yrs_a = [y for y, _ in c['entries_a']]
    yrs_b = [y for y, _ in c['entries_b']]
    print(f"  [{c['score']}] '{c['norm_a']}' {yrs_a} <-> '{c['norm_b']}' {yrs_b}")

## 7. Interactive review of fuzzy matches

For each proposed fuzzy match, you decide:
- **`y`** = Yes, these are the same municipality (merge them)
- **`n`** = No, these are different municipalities (keep separate)
- **`s`** = Skip remaining (stop review, keep rest unmatched)

The original (non-normalized) names are shown so you can judge typos vs. real differences.

In [ ]:
# ------------------------------------------------------------------
# 7. Interactive fuzzy-match review
# ------------------------------------------------------------------
accepted_merges = []  # list of (norm_a, norm_b) pairs to merge

print(f"Review {len(fuzzy_candidates)} fuzzy match candidates.")
print(f"Enter: y=yes (merge), n=no (keep separate), s=stop review\n")

for i, cand in enumerate(fuzzy_candidates):
    origs_a = [f"{o} ({y})" for y, o in cand['entries_a']]
    origs_b = [f"{o} ({y})" for y, o in cand['entries_b']]

    print(f"--- Candidate {i+1}/{len(fuzzy_candidates)} (score={cand['score']}) ---")
    print(f"  A: {cand['norm_a']}")
    print(f"     Original names: {', '.join(origs_a)}")
    print(f"  B: {cand['norm_b']}")
    print(f"     Original names: {', '.join(origs_b)}")

    while True:
        choice = input("  Match? (y/n/s): ").strip().lower()
        if choice in ('y', 'n', 's'):
            break
        print("  Please enter y, n, or s.")

    if choice == 'y':
        accepted_merges.append((cand['norm_a'], cand['norm_b']))
        print("  -> MERGED\n")
    elif choice == 's':
        print("  -> Stopping review. Remaining candidates left unmatched.")
        break
    else:
        print("  -> KEPT SEPARATE\n")

print(f"\nAccepted {len(accepted_merges)} fuzzy merges.")

## 8. Build the canonical municipality mapping

Combine perfect matches and accepted fuzzy merges into a single mapping:
- Each group of equivalent names gets one **canonical ID** (`muni_id`)
- A **canonical label** is chosen (most common original spelling)

In [ ]:
# ------------------------------------------------------------------
# 8. Union-Find to merge groups transitively
# ------------------------------------------------------------------
class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[ra] = rb

uf = UnionFind()

# Register all normalized names
all_norms = set()
for norm, entries in perfect_groups.items():
    if norm:
        all_norms.add(norm)
        uf.find(norm)

# Accepted fuzzy merges
for norm_a, norm_b in accepted_merges:
    uf.union(norm_a, norm_b)

# Build groups
groups = defaultdict(set)
for norm in all_norms:
    root = uf.find(norm)
    groups[root].add(norm)

# Assign canonical IDs and labels
# canonical label = longest original name from the group (likely most complete spelling)
muni_id_map = {}       # norm_name -> muni_id (integer)
muni_label_map = {}    # muni_id -> canonical_label
muni_variants_map = {} # muni_id -> list of all original name variants

for muni_id, (root, norms) in enumerate(sorted(groups.items()), start=1):
    # Collect all original names for this group
    all_originals = []
    for n in norms:
        if n in perfect_groups:
            all_originals.extend([orig for _, orig in perfect_groups[n]])

    # Pick the most common original as the canonical label
    if all_originals:
        from collections import Counter
        label = Counter(all_originals).most_common(1)[0][0]
    else:
        label = root

    for n in norms:
        muni_id_map[n] = muni_id

    muni_label_map[muni_id] = label
    muni_variants_map[muni_id] = sorted(set(all_originals)) if all_originals else [root]

print(f"Total canonical municipalities: {len(groups)}")
print(f"\nSample mappings:")
for mid in list(muni_label_map.keys())[:10]:
    print(f"  ID {mid}: '{muni_label_map[mid]}' <- variants: {muni_variants_map[mid]}")

## 9. Assemble the panel dataset

Add the canonical municipality ID to each cross-section, then combine into one panel.

The final dataset has:
- `muni_id`: canonical municipality identifier
- `muni_canonical`: canonical municipality name
- `year`: benchmark year (1885, 1897, or 1907)
- All original columns from each cross-section (prefixed with the year if needed)

In [ ]:
# ------------------------------------------------------------------
# 9. Map municipality IDs back to dataframes and stack into panel
# ------------------------------------------------------------------

def add_muni_id(df, year):
    """Add muni_id and muni_canonical columns to a cross-section dataframe."""
    df = df.copy()
    df['muni_id'] = df['_muni_norm'].map(muni_id_map)
    df['muni_canonical'] = df['muni_id'].map(muni_label_map)
    df['year'] = year
    # Drop helper column
    df = df.drop(columns=['_muni_norm'])
    return df

panel_1885 = add_muni_id(df_1885, 1885)
panel_1897 = add_muni_id(df_1897, 1897)
panel_1907 = add_muni_id(df_1907, 1907)

# Stack into long-form panel
panel = pd.concat([panel_1885, panel_1897, panel_1907], ignore_index=True)

# Sort by municipality and year
panel = panel.sort_values(['muni_id', 'year']).reset_index(drop=True)

print(f"Panel shape: {panel.shape}")
print(f"Unique municipalities in panel: {panel['muni_id'].nunique()}")
print(f"\nYear distribution:")
print(panel['year'].value_counts().sort_index())
display(panel.head(15))

## 10. Coverage diagnostics

Check which municipalities appear in 1, 2, or all 3 years.

In [ ]:
# ------------------------------------------------------------------
# 10. Coverage diagnostics
# ------------------------------------------------------------------
coverage = panel.groupby('muni_id')['year'].apply(lambda x: sorted(x.unique().tolist()))
n_years = coverage.apply(len)

print("Municipality coverage across years:")
print(f"  In all 3 years: {(n_years == 3).sum()}")
print(f"  In 2 years:     {(n_years == 2).sum()}")
print(f"  In 1 year only: {(n_years == 1).sum()}")

# Municipalities in only 1 year (potential missed matches)
single_year = coverage[n_years == 1]
if len(single_year) > 0:
    print(f"\nMunicipalities appearing in only 1 year (first 30):")
    for mid in single_year.index[:30]:
        label = muni_label_map.get(mid, '?')
        years = single_year[mid]
        print(f"  {label} -> {years}")

## 11. Export the municipality concordance table

Save the mapping of all name variants to canonical IDs, so you can audit or reuse it.

In [ ]:
# ------------------------------------------------------------------
# 11. Export concordance table
# ------------------------------------------------------------------
concordance_rows = []
for mid, variants in muni_variants_map.items():
    for v in variants:
        concordance_rows.append({
            'muni_id': mid,
            'muni_canonical': muni_label_map[mid],
            'name_variant': v
        })

concordance = pd.DataFrame(concordance_rows)
concordance.to_csv('municipality_concordance.csv', index=False)
print(f"Concordance table saved: {concordance.shape[0]} rows")
display(concordance.head(20))

## 12. Export the final panel

Download as CSV and/or Excel.

In [ ]:
# ------------------------------------------------------------------
# 12. Export panel
# ------------------------------------------------------------------

# CSV
panel.to_csv('bank_offices_panel_1885_1897_1907.csv', index=False)
print("Saved: bank_offices_panel_1885_1897_1907.csv")

# Excel
panel.to_excel('bank_offices_panel_1885_1897_1907.xlsx', index=False)
print("Saved: bank_offices_panel_1885_1897_1907.xlsx")

# Download
files.download('bank_offices_panel_1885_1897_1907.csv')
files.download('bank_offices_panel_1885_1897_1907.xlsx')
files.download('municipality_concordance.csv')

print("\nDone! Three files downloaded:")
print("  1. bank_offices_panel_1885_1897_1907.csv  (the panel)")
print("  2. bank_offices_panel_1885_1897_1907.xlsx (the panel, Excel)")
print("  3. municipality_concordance.csv           (name variant mapping)")

## 13. (Optional) Re-run fuzzy matching with a lower threshold

If the diagnostics above show many single-year municipalities that you suspect should
be matched, lower the threshold and re-run. This cell re-does fuzzy matching only on
the still-unmatched names.

In [ ]:
# ------------------------------------------------------------------
# 13. Optional: re-run fuzzy matching at lower threshold
# ------------------------------------------------------------------
NEW_THRESHOLD = int(input("Enter new fuzzy threshold (e.g. 60, or 0 to skip): "))

if NEW_THRESHOLD > 0:
    # Find still-unmatched municipalities (appear in only 1 year)
    single_year_mids = set(n_years[n_years == 1].index)
    # Get their normalized names
    still_unmatched_norms = set()
    for norm, mid in muni_id_map.items():
        if mid in single_year_mids:
            still_unmatched_norms.add(norm)

    print(f"Re-running fuzzy matching on {len(still_unmatched_norms)} unmatched names "
          f"(threshold={NEW_THRESHOLD})\n")

    retry_candidates = []
    seen_pairs = set()

    for norm_a in still_unmatched_norms:
        entries_a = perfect_groups.get(norm_a, [])
        years_a = set(y for y, _ in entries_a)
        targets = [n for n in still_unmatched_norms
                   if n != norm_a and set(y for y, _ in perfect_groups.get(n, [])) != years_a]
        if not targets:
            continue

        matches = process.extract(norm_a, targets, scorer=fuzz.ratio, limit=3)
        for match_name, score, _ in matches:
            if score >= NEW_THRESHOLD:
                pair = tuple(sorted([norm_a, match_name]))
                if pair not in seen_pairs:
                    seen_pairs.add(pair)
                    retry_candidates.append({
                        'norm_a': norm_a,
                        'entries_a': entries_a,
                        'norm_b': match_name,
                        'entries_b': perfect_groups.get(match_name, []),
                        'score': score
                    })

    retry_candidates.sort(key=lambda x: x['score'], reverse=True)
    print(f"Found {len(retry_candidates)} new candidates.\n")

    new_merges = []
    for i, cand in enumerate(retry_candidates):
        origs_a = [f"{o} ({y})" for y, o in cand['entries_a']]
        origs_b = [f"{o} ({y})" for y, o in cand['entries_b']]
        print(f"--- {i+1}/{len(retry_candidates)} (score={cand['score']}) ---")
        print(f"  A: {cand['norm_a']} -> {', '.join(origs_a)}")
        print(f"  B: {cand['norm_b']} -> {', '.join(origs_b)}")

        while True:
            choice = input("  Match? (y/n/s): ").strip().lower()
            if choice in ('y', 'n', 's'):
                break
        if choice == 'y':
            new_merges.append((cand['norm_a'], cand['norm_b']))
            print("  -> MERGED\n")
        elif choice == 's':
            print("  -> Stopping.")
            break
        else:
            print("  -> KEPT SEPARATE\n")

    if new_merges:
        # Apply new merges to Union-Find and rebuild
        for a, b in new_merges:
            uf.union(a, b)

        # Rebuild groups and maps
        groups = defaultdict(set)
        for norm in all_norms:
            root = uf.find(norm)
            groups[root].add(norm)

        muni_id_map.clear()
        muni_label_map.clear()
        muni_variants_map.clear()

        for muni_id, (root, norms) in enumerate(sorted(groups.items()), start=1):
            all_originals = []
            for n in norms:
                if n in perfect_groups:
                    all_originals.extend([orig for _, orig in perfect_groups[n]])
            if all_originals:
                from collections import Counter
                label = Counter(all_originals).most_common(1)[0][0]
            else:
                label = root
            for n in norms:
                muni_id_map[n] = muni_id
            muni_label_map[muni_id] = label
            muni_variants_map[muni_id] = sorted(set(all_originals)) if all_originals else [root]

        # Rebuild panel
        panel_1885 = add_muni_id(df_1885, 1885)
        panel_1897 = add_muni_id(df_1897, 1897)
        panel_1907 = add_muni_id(df_1907, 1907)
        panel = pd.concat([panel_1885, panel_1897, panel_1907], ignore_index=True)
        panel = panel.sort_values(['muni_id', 'year']).reset_index(drop=True)

        # Re-export
        panel.to_csv('bank_offices_panel_1885_1897_1907.csv', index=False)
        panel.to_excel('bank_offices_panel_1885_1897_1907.xlsx', index=False)
        print(f"\nUpdated panel: {panel.shape[0]} rows, {panel['muni_id'].nunique()} municipalities")
        files.download('bank_offices_panel_1885_1897_1907.csv')
    else:
        print("No new merges accepted.")
else:
    print("Skipped.")